In [1]:
import nltk

nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Dell\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
import pandas as pd
# 1. LOAD DATA
df = pd.read_csv("ai_teacher_results.csv")

# 2. PREPARE REFERENCE AND GENERATED ANSWERS
references = (
    df["Teacher Response"]
    .fillna("")
    .astype(str)
    .tolist()
)
candidates = (
    df["GeneratedAnswer"]
    .fillna("")
    .astype(str)
    .tolist()
)

# 3. BERTSCORE
P, R, F1 = bert_score(
    candidates,
    references,
    lang="en",
    verbose=True
)
df["BERT"] = F1.cpu().numpy()

# 4. ROUGE
rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"],use_stemmer=True)
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for candidate, reference in zip(candidates, references):
    scores = rouge.score(reference, candidate)
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

df["ROUGE-1"] = rouge1_scores
df["ROUGE-2"] = rouge2_scores
df["ROUGE-L"] = rougeL_scores

# 5. METEOR
meteor_scores = []

for candidate, reference in zip(candidates, references):
    candidate_tokens = candidate.split()
    reference_tokens = reference.split()
    score = meteor_score([reference_tokens], candidate_tokens)
    meteor_scores.append(score)

df["METEOR"] = meteor_scores

# 6. ROUND SCORES

similarity_columns = [
    "BERT",
    "ROUGE-1",
    "ROUGE-2",
    "ROUGE-L",
    "METEOR"
]
df[similarity_columns] = (df[similarity_columns].round(3))

# 7. QUESTION-WISE SIMILARITY DATAFRAME

similarity_result = df[
    [
        "User Query",
        "BERT",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "METEOR"
    ]
].copy()

# DISPLAY QUESTION-WISE DATAFRAME

display(similarity_result)

# 8. CALCULATE AVERAGES

average_bert = df["BERT"].mean()
average_rouge1 = df["ROUGE-1"].mean()
average_rouge2 = df["ROUGE-2"].mean()
average_rougeL = df["ROUGE-L"].mean()
average_meteor = df["METEOR"].mean()

# 9. CREATE AVERAGE SIMILARITY DATAFRAME

average_similarity = pd.DataFrame({
    "Metric": ["BERT", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR"],
    "Average Score": [average_bert, average_rouge1, average_rouge2, average_rougeL, average_meteor]
})

# ROUND AVERAGE SCORES
average_similarity["Average Score"] = (average_similarity["Average Score"].round(3))

# DISPLAY AVERAGE DATAFRAME
display(average_similarity)

C:\Users\Dell\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Dell\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.42GB            

model.safetensors: downloading bytes:           |  0.00B            

In [ ]:
#!pip install bert_score

In [ ]:
#!pip install rouge_score